In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [1]:
import io
import torch
from pathlib import Path

path = Path.cwd().joinpath("models")
path.mkdir(exist_ok=True)
file = path.joinpath("full_model.pt")
?path.rmdir

Signature: path.rmdir()
Docstring: Remove this directory.  The directory must be empty.
File:      ~/miniconda3/envs/kiwi/lib/python3.9/pathlib.py
Type:      method


In [2]:
from linodenet.models import (
    LinODEnet,
    LinODECell,
    LinearContraction,
    iResNet,
    iResNetBlock,
)

model = LinearContraction(32, 32);

In [3]:
model.spectral_norm

tensor(1.1196, grad_fn=<CopyBackwards>)

In [4]:
model(torch.randn(32))

tensor([ 0.0520, -0.5487,  0.1102,  0.7100,  0.6251, -1.0486, -0.7608, -0.1791,
         0.6580, -0.2608, -0.6204, -0.1340,  1.3023,  0.3596, -0.1874, -1.2044,
         0.9739,  0.5494, -0.4135, -0.2674, -0.7123, -0.1399,  0.6563,  0.4977,
         0.1711,  1.4023,  0.0483,  0.2423,  0.0258, -0.8398, -0.3588, -0.0116],
       grad_fn=<AddBackward0>)

In [5]:
model.spectral_norm

tensor(1.1196, grad_fn=<CopyBackwards>)

In [8]:
torch.linalg.matrix_norm(model.weight, ord=2)

tensor(1.1196, grad_fn=<CopyBackwards>)

In [20]:
model = LinODEnet(32, 32)

model.state_dict()

OrderedDict([('kernel',
              tensor([[ 0.0000, -0.0511,  0.1741,  ..., -0.1968, -0.0674,  0.2747],
                      [ 0.0511,  0.0000,  0.0144,  ..., -0.2795,  0.2223, -0.2572],
                      [-0.1741, -0.0144,  0.0000,  ...,  0.1862, -0.1570,  0.0536],
                      ...,
                      [ 0.1968,  0.2795, -0.1862,  ...,  0.0000,  0.1339,  0.2597],
                      [ 0.0674, -0.2223,  0.1570,  ..., -0.1339,  0.0000,  0.0709],
                      [-0.2747,  0.2572, -0.0536,  ..., -0.2597, -0.0709,  0.0000]])),
             ('ZERO', tensor(0.)),
             ('encoder.blocks.0.bottleneck.0.weight',
              tensor([[ 2.7225e-02,  1.0740e-01, -1.2867e-01,  1.2747e-01,  9.6937e-02,
                       -1.3068e-01, -1.7283e-01,  7.7208e-02, -5.9941e-02, -2.1518e-04,
                       -1.6722e-01,  2.9242e-02, -1.5679e-04,  5.8364e-02, -1.5544e-01,
                       -1.0102e-01,  1.1666e-01, -1.5857e-01,  7.0662e-03,  1.0014e-01,
 

# Try save / load with BytesIO

In [20]:
buffer = io.BytesIO()
torch.jit.save(model, buffer)

In [21]:
buffer.seek(0)
new_m = torch.jit.load(buffer)

RuntimeError: 

In [27]:
del model

In [28]:
torch.jit.load(file)

RuntimeError: 

In [5]:
# Load ScriptModule from io.BytesIO object
with open(path.joinpath("full_model.pt"), "rb") as file:
    buffer = io.BytesIO(file.read())

torch.jit.load(buffer)

RuntimeError: 

In [6]:
# Load all tensors to the original device
torch.jit.load(buffer)

RuntimeError: PytorchStreamReader failed reading zip archive: not a ZIP archive

In [7]:
# Load all tensors onto CPU, using a device
buffer.seek(0)
torch.jit.load(buffer, map_location=torch.device("cpu"))

RuntimeError: 

In [8]:
# Load all tensors onto CPU, using a string
buffer.seek(0)
torch.jit.load(buffer, map_location="cpu")

RuntimeError: 